In [41]:
import ipyparallel as ipp
n = 10
cluster = ipp.Cluster(engines = "mpi", n = n)
rc = cluster.start_and_connect_sync()
rc.activate()
view = rc[:]

Starting 10 engines with <class 'ipyparallel.cluster.launcher.MPIEngineSetLauncher'>


  0%|          | 0/10 [00:00<?, ?engine/s]

In [64]:
%%px 

import logging
from pathlib import Path
import subprocess
from mpi4py import MPI
import dolfinx
import adios4dolfinx
import time 
import os 
from dolfinx.io import gmshio
from dolfinx import fem, io, mesh
import gmsh 
import numpy as np


In [57]:
%%px

def load_mesh(msh_file):
    # Convert mesh if needed and import
    print(type(msh_file))
    mesh_, cell_tags, facet_tags = gmshio.read_from_msh(
        msh_file, 
        comm = MPI.COMM_WORLD, 
        rank=0, 
        gdim=3, 
        partitioner = dolfinx.mesh.create_cell_partitioner(mesh.GhostMode.shared_facet)
    )
    return mesh_, cell_tags, facet_tags


def write_partitioned_mesh(meshfile: Path, savedir: Path):
    ''' 
    Input: 
        meshfile: Path to the mesh file
        filename: Path to the output folder
    Output:
        - None
    Action: 
        - Read the mesh file
        - Write the mesh to a partitioned file using ADIOS2
    '''
    meshfile = Path(meshfile)
    savedir = Path(savedir)

    name_part = meshfile.stem # Get the name of the file without the extension
    number_str = name_part.split('_')[-1]  # get the number of the mesh

    savedir.mkdir(parents=True, exist_ok=True)  
    save_path = savedir / f"{number_str}.bp"

    mesh, cell_tags, facet_tags = load_mesh(meshfile)

    # Write mesh checkpoint
    adios4dolfinx.write_mesh(save_path, mesh, engine="BP4", store_partition_info=True)
    adios4dolfinx.write_meshtags(save_path, mesh, cell_tags, engine="BP4", meshtag_name = "cells")
    adios4dolfinx.write_meshtags(save_path, mesh, facet_tags, engine="BP4", meshtag_name = "facets")
    return



def read_partitioned_mesh(filename: Path, read_from_partition: bool = True):
    prefix = f"{MPI.COMM_WORLD.rank + 1}/{MPI.COMM_WORLD.size}: "
    try:
        mesh = adios4dolfinx.read_mesh(
            filename, comm=MPI.COMM_WORLD, engine="BP4", read_from_partition=read_from_partition
        )
        cell_tags = adios4dolfinx.read_meshtags(filename, mesh, meshtag_name = "cells", engine="BP4")
        facet_tags = adios4dolfinx.read_meshtags(filename, mesh, meshtag_name = "facets", engine="BP4")

        tdim = mesh.topology.dim
        mesh.topology.create_connectivity(tdim - 1, tdim)

        print(f"{prefix} Mesh: {mesh.name} read successfully with {read_from_partition=}")
    except ValueError as e:
        print(f"{prefix} Caught exception: ", e)

    return mesh, cell_tags, facet_tags

In [58]:
%%px

mesh_save_path = Path("partitioned_meshes")

mesh_folder_path = Path("superlayers/superlayer_00005.msh")
write_partitioned_mesh(mesh_folder_path, mesh_save_path)
mesh_folder_path = Path("superlayers/superlayer_00006.msh")
write_partitioned_mesh(mesh_folder_path, mesh_save_path)

[stdout:0] <class 'pathlib._local.PosixPath'>
Info    : Reading 'superlayers/superlayer_00005.msh'...
Info    : 54 entities
Info    : 501552 nodes
Info    : 2989261 elements
Info    : Done reading 'superlayers/superlayer_00005.msh'
<class 'pathlib._local.PosixPath'>
Info    : Reading 'superlayers/superlayer_00006.msh'...
Info    : 54 entities
Info    : 503526 nodes
Info    : 3005955 elements
Info    : Done reading 'superlayers/superlayer_00006.msh'


[stdout:1] <class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>


[stdout:2] <class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>


[stdout:6] <class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>


[stdout:4] <class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>


...

[stdout:8] <class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>


[stdout:7] <class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>


[stdout:5] <class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>


[stdout:9] <class 'pathlib._local.PosixPath'>
<class 'pathlib._local.PosixPath'>


%px:   0%|          | 0/10 [00:00<?, ?tasks/s]

In [62]:
%%px 

import time 

mesh_file = Path("partitioned_meshes/00005.bp")
#write_partitioned_mesh(mesh_file)

start_time = time.time()
mesh1, cell_tags1, facet_tags1 = read_partitioned_mesh(mesh_file, True)
end_time = time.time()
print(f"Time taken to read mesh: {(end_time - start_time)*1000} ms")
print(f"Loaded mesh with {mesh1.topology.index_map(3).size_local} cells.")
print(len(cell_tags1.values), "DOFs for this process")

mesh_file = Path("partitioned_meshes/00006.bp")
start_time = time.time()
mesh2, cell_tags2, facet_tags2 = read_partitioned_mesh(mesh_file, False)
end_time = time.time()
print(f"Time taken to read mesh: {(end_time - start_time)*1000} ms")
print(f"Loaded mesh with {mesh2.topology.index_map(3).size_local} cells.")
print(len(cell_tags2.values), "DOFs for this process")

%px:   0%|          | 0/10 [00:00<?, ?tasks/s]

[stdout:6] 7/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 2663.775682449341 ms
Loaded mesh with 272484 cells.
274225 DOFs for this process
7/10:  Mesh: mesh read successfully with read_from_partition=False
Time taken to read mesh: 3670.542001724243 ms
Loaded mesh with 279363 cells.
282810 DOFs for this process


[stdout:8] 9/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 2709.840774536133 ms
Loaded mesh with 275686 cells.
279318 DOFs for this process
9/10:  Mesh: mesh read successfully with read_from_partition=False
Time taken to read mesh: 3623.6555576324463 ms
Loaded mesh with 280239 cells.
283825 DOFs for this process


[stdout:1] 2/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 2697.406530380249 ms
Loaded mesh with 275372 cells.
278238 DOFs for this process
2/10:  Mesh: mesh read successfully with read_from_partition=False
Time taken to read mesh: 3612.842559814453 ms
Loaded mesh with 279293 cells.
283150 DOFs for this process


[stdout:3] 4/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 2709.43021774292 ms
Loaded mesh with 273449 cells.
276997 DOFs for this process
4/10:  Mesh: mesh read successfully with read_from_partition=False
Time taken to read mesh: 3614.5989894866943 ms
Loaded mesh with 272114 cells.
276913 DOFs for this process


...

[stdout:4] 5/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 2702.3215293884277 ms
Loaded mesh with 273259 cells.
277013 DOFs for this process
5/10:  Mesh: mesh read successfully with read_from_partition=False
Time taken to read mesh: 3616.333484649658 ms
Loaded mesh with 279558 cells.
282266 DOFs for this process


[stdout:0] 1/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 2717.55051612854 ms
Loaded mesh with 273049 cells.
275404 DOFs for this process
1/10:  Mesh: mesh read successfully with read_from_partition=False
Time taken to read mesh: 3543.785572052002 ms
Loaded mesh with 273180 cells.
275874 DOFs for this process


[stdout:7] 8/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 2703.077554702759 ms
Loaded mesh with 273130 cells.
274771 DOFs for this process
8/10:  Mesh: mesh read successfully with read_from_partition=False
Time taken to read mesh: 3628.3740997314453 ms
Loaded mesh with 274338 cells.
276212 DOFs for this process


[stdout:2] 3/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 2664.3967628479004 ms
Loaded mesh with 273461 cells.
275896 DOFs for this process
3/10:  Mesh: mesh read successfully with read_from_partition=False
Time taken to read mesh: 3647.3376750946045 ms
Loaded mesh with 279544 cells.
282117 DOFs for this process


[stdout:5] 6/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 2700.4635334014893 ms
Loaded mesh with 275484 cells.
279098 DOFs for this process
6/10:  Mesh: mesh read successfully with read_from_partition=False
Time taken to read mesh: 3640.7997608184814 ms
Loaded mesh with 266469 cells.
269260 DOFs for this process


In [93]:
%%px 

start_time = time.time()
mask_00005 = np.load("partitioned_meshes/mask_00005.npy")
end_time = time.time()
print(f"Time taken to read mask: {(end_time - start_time)*1000} ms")

start_time = time.time()
mask_00006 = np.load("partitioned_meshes/mask_00006.npy")
end_time = time.time()
print(f"Time taken to read mask: {(end_time - start_time)*1000} ms")

[stdout:0] Time taken to read mask: 0.6196498870849609 ms
Time taken to read mask: 0.3387928009033203 ms


[stdout:2] Time taken to read mask: 0.4544258117675781 ms
Time taken to read mask: 0.35500526428222656 ms


[stdout:1] Time taken to read mask: 1.1932849884033203 ms
Time taken to read mask: 0.2999305725097656 ms


[stdout:3] Time taken to read mask: 0.5166530609130859 ms
Time taken to read mask: 0.3559589385986328 ms


[stdout:5] Time taken to read mask: 0.9143352508544922 ms
Time taken to read mask: 0.4737377166748047 ms


[stdout:4] Time taken to read mask: 0.6322860717773438 ms
Time taken to read mask: 0.3821849822998047 ms


[stdout:6] Time taken to read mask: 0.5743503570556641 ms
Time taken to read mask: 0.4744529724121094 ms


[stdout:9] Time taken to read mask: 0.5548000335693359 ms
Time taken to read mask: 0.24628639221191406 ms


[stdout:8] Time taken to read mask: 0.5755424499511719 ms
Time taken to read mask: 0.3478527069091797 ms


[stdout:7] Time taken to read mask: 0.5948543548583984 ms
Time taken to read mask: 0.4966259002685547 ms


In [ ]:

for filename in sorted(os.listdir(folder_path)):
    write_partitioned_mesh(filename, mesh_save_path)

    full_path = os.path.join(folder_path, filename)
    mesh_file = Path("partitioned_mesh.bp")
    write_partitioned_mesh(mesh_file)
    print("File:", full_path)
    break

In [ ]:
%%px 

mesh_file = Path("partitioned_mesh.bp")
#write_partitioned_mesh(mesh_file)

start_time = time.time()
mesh2, cell_tags2, facet_tags2 = read_partitioned_mesh(mesh_file, True)
end_time = time.time()
print(f"Time taken to read mesh: {(end_time - start_time)*1000} ms")
print(f"Loaded mesh with {mesh1.topology.index_map(3).size_local} cells.")
#print(len(mesh.cell_tags.values), "DOFs for this process")